# 03 — Integrated Gradients

Integrated Gradients (IG) provides attribution for neural networks with theoretical guarantees.

## Path Integral and the Completeness Axiom

IG attributes a neural net's prediction to its inputs by integrating the gradient along a straight-line path from a baseline input *x'* (e.g., all zeros) to the actual input *x*:

$$\text{IG}_i(x) = (x_i - x'_i) \int_0^1 \frac{\partial F(x' + \alpha(x - x')}{\partial x_i} d\alpha$$

The integral is approximated by summing gradients at *m* equally spaced points along the path.

**Completeness axiom**: The attributions sum exactly to the difference in predictions:
$$\sum_i \text{IG}_i(x) = F(x) - F(x')$$

This distinguishes IG from vanilla gradients, which fail completeness. IG also satisfies **sensitivity** (a feature that changes the output gets non-zero attribution) and **implementation invariance** (equivalent networks produce identical attributions).

The choice of baseline matters: a black image is natural for vision, a zero vector for text embeddings, or the mean training sample for tabular data.

In [ ]:
# Integrated Gradients on a neural classifier
import numpy as np
import torch
import torch.nn as nn

# Small neural net for tabular classification
class TabNet(nn.Module):
    def __init__(self, n_features, n_hidden=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, n_hidden),
            nn.ReLU(),
            nn.Linear(n_hidden, n_hidden),
            nn.ReLU(),
            nn.Linear(n_hidden, 1),
        )
    def forward(self, x):
        return torch.sigmoid(self.net(x)).squeeze(-1)

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X, y = make_classification(n_samples=1000, n_features=10, n_informative=5, random_state=42)
scaler = StandardScaler()
X = scaler.fit_transform(X)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

Xtr = torch.tensor(X_tr, dtype=torch.float32)
ytr = torch.tensor(y_tr, dtype=torch.float32)
Xte = torch.tensor(X_te, dtype=torch.float32)

model = TabNet(10)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
for epoch in range(50):
    model.train()
    pred = model(Xtr)
    loss = nn.functional.binary_cross_entropy(pred, ytr)
    opt.zero_grad(); loss.backward(); opt.step()

model.eval()
with torch.no_grad():
    acc = ((model(Xte) > 0.5).float() == torch.tensor(y_te, dtype=torch.float32)).float().mean()
print(f'Test accuracy: {acc:.3f}')

In [ ]:
# Integrated Gradients implementation
def integrated_gradients(model, x, baseline=None, n_steps=50):
    """
    Compute IG attributions.
    x: (n_features,) tensor
    baseline: same shape; defaults to zeros
    """
    if baseline is None:
        baseline = torch.zeros_like(x)

    # Interpolate from baseline to input
    alphas = torch.linspace(0, 1, n_steps + 1)
    interpolated = baseline + alphas.unsqueeze(1) * (x - baseline).unsqueeze(0)
    interpolated.requires_grad_(True)

    # Forward pass
    preds = model(interpolated)  # (n_steps+1,)

    # Aggregate gradients via autograd
    grads = []
    for i in range(len(alphas)):
        if interpolated.grad is not None:
            interpolated.grad.zero_()
        preds[i].backward(retain_graph=True)
        grads.append(interpolated.grad[i].clone())

    grads = torch.stack(grads)  # (n_steps+1, n_features)
    # Trapezoidal rule
    avg_grads = (grads[:-1] + grads[1:]) / 2
    ig = (x - baseline) * avg_grads.mean(0)
    return ig

x_sample = Xte[0].clone()
baseline = torch.zeros(10)
attrs = integrated_gradients(model, x_sample, baseline, n_steps=100)

print('Integrated Gradients attributions:')
for i, v in enumerate(attrs.detach()):
    print(f'  f{i}: {v:.4f}')

# Completeness check
with torch.no_grad():
    f_x = model(x_sample.unsqueeze(0)).item()
    f_base = model(baseline.unsqueeze(0)).item()
print(f'\nCompleteness: F(x)-F(baseline) = {f_x - f_base:.4f}')
print(f'Sum of IGs: {attrs.detach().sum():.4f}')

In [ ]:
# Visualise IG attributions
import matplotlib.pyplot as plt

ig_vals = attrs.detach().numpy()
feature_names = [f'f{i}' for i in range(10)]
sorted_idx = np.argsort(np.abs(ig_vals))[::-1]

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['steelblue' if v > 0 else 'tomato' for v in ig_vals[sorted_idx]]
ax.barh([feature_names[i] for i in sorted_idx], ig_vals[sorted_idx], color=colors)
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('IG attribution')
ax.set_title('Integrated Gradients — Sample 0')
plt.tight_layout()
plt.savefig('/tmp/ig_attribution.png', dpi=100, bbox_inches='tight')
plt.show()
print('IG attribution plot saved')